# 🔍 Anomaly Detection from Sensor Readings
### CEIP DS Lloyd Kaggle Competition
**Objective:** Predict anomalies (target=1) vs normal readings (target=0) from 5 sensor features (X1–X5) and a timestamp.

---
**Notebook Structure:**
1. Data Loading & Exploration
2. Preprocessing & Feature Engineering
3. Classical Models
4. Advanced / Ensemble Models
5. Neural Network
6. Model Comparison & Final Submission

In [ ]:
# ─── Core Libraries ───────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# ─── Sklearn ──────────────────────────────────────────────────────────────────
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, classification_report, confusion_matrix,
                             ConfusionMatrixDisplay, roc_auc_score, roc_curve)
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline

# ─── Advanced Models ──────────────────────────────────────────────────────────
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

# ─── Imbalanced Learning ──────────────────────────────────────────────────────
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

# ─── Deep Learning ────────────────────────────────────────────────────────────
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

print('All libraries loaded successfully ✅')
print(f'TensorFlow version: {tf.__version__}')

## 1. Data Loading & Exploration

In [ ]:
# Load data
train = pd.read_parquet('train.parquet')
test  = pd.read_parquet('test.parquet')

print('Train shape:', train.shape)
print('Test shape :', test.shape)
train.head(5)

In [ ]:
# Basic info
print('=== DATA TYPES ===')
print(train.dtypes)
print('\n=== MISSING VALUES ===')
print(train.isnull().sum())
print('\n=== STATISTICAL SUMMARY ===')
train.describe()

In [ ]:
# ─── Class Distribution ───────────────────────────────────────────────────────
vc = train['target'].value_counts()
anomaly_pct = vc['1'] / len(train) * 100

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar chart
axes[0].bar(['Normal (0)', 'Anomaly (1)'], vc.values, color=['#2ecc71', '#e74c3c'], edgecolor='black')
axes[0].set_title('Class Distribution (Count)', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Count')
for i, v in enumerate(vc.values):
    axes[0].text(i, v + 5000, f'{v:,}', ha='center', fontweight='bold')

# Pie chart
axes[1].pie(vc.values, labels=['Normal (0)', 'Anomaly (1)'],
            colors=['#2ecc71', '#e74c3c'], autopct='%1.2f%%',
            startangle=90, explode=[0, 0.1])
axes[1].set_title(f'Class Balance\nAnomaly = {anomaly_pct:.2f}% of data', fontsize=13, fontweight='bold')

plt.suptitle('⚠️  Highly Imbalanced Dataset', fontsize=14, color='red', y=1.02)
plt.tight_layout()
plt.savefig('class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Normal  : {vc["0"]:,} ({100-anomaly_pct:.2f}%)')
print(f'Anomaly : {vc["1"]:,} ({anomaly_pct:.2f}%)')
print('⚠️  Severe class imbalance — we must use SMOTE + class_weight strategies')

In [ ]:
# ─── Feature Distributions ────────────────────────────────────────────────────
features = ['X1', 'X2', 'X3', 'X4', 'X5']

fig, axes = plt.subplots(2, 5, figsize=(18, 7))
colors = {'0': '#2ecc71', '1': '#e74c3c'}

for i, feat in enumerate(features):
    for target_val in ['0', '1']:
        subset = train[train['target'] == target_val][feat]
        # Use log scale for features with extreme range
        log_vals = np.log1p(np.clip(subset, 0, None))
        axes[0, i].hist(log_vals, bins=50, alpha=0.6,
                        color=colors[target_val],
                        label=f'target={target_val}', density=True)
    axes[0, i].set_title(f'{feat} (log scale)', fontweight='bold')
    axes[0, i].legend(fontsize=8)
    axes[0, i].set_xlabel('log1p value')

# Box plots
for i, feat in enumerate(features):
    data_plot = [np.log1p(np.clip(train[train['target']=='0'][feat], 0, None)),
                 np.log1p(np.clip(train[train['target']=='1'][feat], 0, None))]
    bp = axes[1, i].boxplot(data_plot, labels=['Normal', 'Anomaly'],
                             patch_artist=True)
    bp['boxes'][0].set_facecolor('#2ecc71')
    bp['boxes'][1].set_facecolor('#e74c3c')
    axes[1, i].set_title(f'{feat} Boxplot', fontweight='bold')
    axes[1, i].set_ylabel('log1p value')

plt.suptitle('Feature Distributions by Class (Log-Transformed)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. Preprocessing & Feature Engineering

In [ ]:
# ─── Observations from EDA ────────────────────────────────────────────────────
# X3, X4 contain exponential-scale values (e^n) → log-transform
# X4 has extreme outliers up to 5.5e34
# Date is a datetime → extract temporal features
# No missing values → no imputation needed

def engineer_features(df):
    df = df.copy()

    # ── Temporal features from Date ──────────────────────────────────────────
    df['year']        = df['Date'].dt.year
    df['month']       = df['Date'].dt.month
    df['day_of_year'] = df['Date'].dt.dayofyear
    df['quarter']     = df['Date'].dt.quarter
    df['day_of_week'] = df['Date'].dt.dayofweek   # 0=Mon … 6=Sun
    df['is_weekend']  = (df['day_of_week'] >= 5).astype(int)

    # ── Log-transform skewed features ────────────────────────────────────────
    # X3, X4 are exponential-scale → log makes them linear-friendly
    df['X3_log'] = np.log1p(np.clip(df['X3'], 0, None))
    df['X4_log'] = np.log1p(np.clip(df['X4'], 0, None))

    # ── Interaction features ──────────────────────────────────────────────────
    df['X1_X2']      = df['X1'] * df['X2']        # sensor product
    df['X1_X5']      = df['X1'] * df['X5']        # sensor product
    df['X2_X5']      = df['X2'] * df['X5']
    df['X1_minus_X5']= df['X1'] - df['X5']        # sensor difference
    df['X2_div_X1']  = df['X2'] / (df['X1'] + 1e-9)

    # ── Ratio / normalised features ───────────────────────────────────────────
    df['X4_log_div_X3_log'] = df['X4_log'] / (df['X3_log'] + 1e-9)

    # ── Magnitude / threshold flags ───────────────────────────────────────────
    df['X4_extreme'] = (df['X4'] > 1e10).astype(int)   # flag extreme X4
    df['X3_extreme'] = (df['X3'] > 1e15).astype(int)   # flag extreme X3
    df['X5_zero']    = (df['X5'] == 0).astype(int)     # X5 = 0 indicator

    return df

train_fe = engineer_features(train)
test_fe  = engineer_features(test)

print('Features after engineering:', train_fe.shape[1])
print('New columns added:', [c for c in train_fe.columns if c not in train.columns])

In [ ]:
# ─── Correlation Heatmap ──────────────────────────────────────────────────────
numeric_cols = train_fe.select_dtypes(include=[np.number]).columns.tolist()
# Add binary target for correlation
corr_df = train_fe[numeric_cols].copy()
corr_df['target_num'] = train_fe['target'].astype(int)

corr = corr_df.corr()

plt.figure(figsize=(16, 12))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, linewidths=0.5, annot_kws={'size': 8})
plt.title('Feature Correlation Heatmap\n(Including Engineered Features)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

# Top correlations with target
target_corr = corr['target_num'].drop('target_num').abs().sort_values(ascending=False)
print('\nTop features correlated with target:')
print(target_corr.head(10))

In [ ]:
# ─── Prepare X, y ─────────────────────────────────────────────────────────────
DROP_COLS = ['Date', 'target']   # train only
TEST_DROP = ['Date', 'ID']       # test only

FEATURE_COLS = [c for c in train_fe.columns if c not in DROP_COLS]

X = train_fe[FEATURE_COLS].values
y = train_fe['target'].astype(int).values

X_test_final = test_fe[[c for c in FEATURE_COLS if c in test_fe.columns]].values
test_ids     = test_fe['ID'].values

# Stratified split to preserve class ratio
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

print(f'Train: {X_train.shape}, Val: {X_val.shape}')
print(f'Train anomaly %: {y_train.mean()*100:.2f}%')
print(f'Val   anomaly %: {y_val.mean()*100:.2f}%')
print(f'Feature columns ({len(FEATURE_COLS)}):', FEATURE_COLS)

In [ ]:
# ─── Scale features ───────────────────────────────────────────────────────────
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_val_sc   = scaler.transform(X_val)
X_test_sc  = scaler.transform(X_test_final)

# ─── SMOTE on training set only ───────────────────────────────────────────────
# Oversample minority class to 10% ratio (full 50/50 may hurt precision)
smote = SMOTE(sampling_strategy=0.1, random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train_sc, y_train)

print(f'After SMOTE — Train size: {X_train_sm.shape}')
print(f'Anomaly count after SMOTE: {y_train_sm.sum():,} ({y_train_sm.mean()*100:.1f}%)')

## 3. Helper: Evaluation Function

In [ ]:
results_log = []   # store all model results

def evaluate_model(name, model, X_tr, y_tr, X_v, y_v, plot=True):
    """Fit model, evaluate on validation set, log results."""
    model.fit(X_tr, y_tr)
    y_pred = model.predict(X_v)

    acc  = accuracy_score(y_v, y_pred)
    prec = precision_score(y_v, y_pred, zero_division=0)
    rec  = recall_score(y_v, y_pred, zero_division=0)
    f1   = f1_score(y_v, y_pred, zero_division=0)
    f1_macro = f1_score(y_v, y_pred, average='macro', zero_division=0)

    results_log.append({
        'Model': name, 'Accuracy': acc,
        'Precision': prec, 'Recall': rec,
        'F1 (anomaly)': f1, 'F1 (macro)': f1_macro
    })

    print(f'\n{'='*50}')
    print(f'  {name}')
    print(f'{'='*50}')
    print(classification_report(y_v, y_pred,
                                target_names=['Normal(0)', 'Anomaly(1)'],
                                zero_division=0))
    print(f'  Macro F1: {f1_macro:.4f}  |  Anomaly F1: {f1:.4f}')

    if plot:
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))

        # Confusion matrix
        cm = confusion_matrix(y_v, y_pred)
        ConfusionMatrixDisplay(cm, display_labels=['Normal','Anomaly']).plot(
            ax=axes[0], colorbar=False, cmap='Blues')
        axes[0].set_title(f'{name}\nConfusion Matrix')

        # ROC curve (if proba available)
        try:
            y_prob = model.predict_proba(X_v)[:, 1]
            fpr, tpr, _ = roc_curve(y_v, y_prob)
            auc = roc_auc_score(y_v, y_prob)
            axes[1].plot(fpr, tpr, color='#e74c3c', lw=2,
                         label=f'AUC = {auc:.3f}')
            axes[1].plot([0,1],[0,1],'--', color='gray')
            axes[1].set_xlabel('False Positive Rate')
            axes[1].set_ylabel('True Positive Rate')
            axes[1].set_title('ROC Curve')
            axes[1].legend()
        except Exception:
            axes[1].text(0.5, 0.5, 'Probabilities\nnot available',
                         ha='center', va='center', transform=axes[1].transAxes)
            axes[1].set_title('ROC Curve (N/A)')

        plt.suptitle(name, fontsize=13, fontweight='bold')
        plt.tight_layout()
        plt.show()

    return model

## 4. Classical Models

In [ ]:
# ─── 4.1 Logistic Regression ──────────────────────────────────────────────────
# Fast, interpretable baseline. class_weight='balanced' handles imbalance.
lr = LogisticRegression(class_weight='balanced', max_iter=1000,
                        C=1.0, solver='lbfgs', random_state=42)
lr = evaluate_model('Logistic Regression', lr, X_train_sm, y_train_sm, X_val_sc, y_val)

In [ ]:
# ─── 4.2 Decision Tree ────────────────────────────────────────────────────────
# Non-linear, interpretable, prone to overfitting → tuned depth
dt = DecisionTreeClassifier(max_depth=10, class_weight='balanced',
                             min_samples_leaf=50, random_state=42)
dt = evaluate_model('Decision Tree', dt, X_train_sm, y_train_sm, X_val_sc, y_val)

In [ ]:
# ─── 4.3 K-Nearest Neighbours ─────────────────────────────────────────────────
# Instance-based; effective on scaled data. Subset used for speed.
# Using a 20% sample of SMOTE data for speed (KNN is O(n) at inference)
idx_sample = np.random.choice(len(X_train_sm), size=int(0.2*len(X_train_sm)), replace=False)
knn = KNeighborsClassifier(n_neighbors=11, metric='euclidean', n_jobs=-1)
knn = evaluate_model('KNN (k=11)', knn,
                     X_train_sm[idx_sample], y_train_sm[idx_sample],
                     X_val_sc, y_val)

In [ ]:
# ─── 4.4 Support Vector Machine ───────────────────────────────────────────────
# Powerful for high-dimensional data; class_weight='balanced' crucial here.
# Using LinearSVC for speed on large dataset
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV

svc_base = LinearSVC(class_weight='balanced', C=0.1, max_iter=2000, random_state=42)
svm = CalibratedClassifierCV(svc_base, cv=3)   # wrap to get predict_proba
svm = evaluate_model('SVM (LinearSVC)', svm, X_train_sm, y_train_sm, X_val_sc, y_val)

## 5. Advanced / Ensemble Models

In [ ]:
# ─── 5.1 Random Forest ────────────────────────────────────────────────────────
# Ensemble of decision trees; robust to overfitting; built-in feature importance
rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=15,
    min_samples_leaf=20,
    class_weight='balanced_subsample',
    n_jobs=-1,
    random_state=42
)
rf = evaluate_model('Random Forest', rf, X_train_sm, y_train_sm, X_val_sc, y_val)

In [ ]:
# ─── Feature Importance from Random Forest ────────────────────────────────────
importances = pd.Series(rf.feature_importances_, index=FEATURE_COLS)
importances = importances.sort_values(ascending=True)

plt.figure(figsize=(10, 8))
importances.plot(kind='barh', color='#3498db', edgecolor='black')
plt.title('Random Forest — Feature Importances', fontsize=13, fontweight='bold')
plt.xlabel('Importance Score')
plt.tight_layout()
plt.savefig('feature_importance_rf.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ─── 5.2 XGBoost ──────────────────────────────────────────────────────────────
# Gradient boosting; scale_pos_weight handles imbalance natively
pos_weight = int((y_train == 0).sum() / (y_train == 1).sum())
print(f'scale_pos_weight = {pos_weight}  (ratio of negatives to positives)')

xgb = XGBClassifier(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    scale_pos_weight=pos_weight,
    subsample=0.8,
    colsample_bytree=0.8,
    use_label_encoder=False,
    eval_metric='logloss',
    tree_method='hist',
    random_state=42,
    n_jobs=-1
)
xgb = evaluate_model('XGBoost', xgb, X_train_sc, y_train, X_val_sc, y_val)

In [ ]:
# ─── 5.3 LightGBM ─────────────────────────────────────────────────────────────
# Fastest gradient boosting; great for large datasets like this one
lgbm = LGBMClassifier(
    n_estimators=500,
    max_depth=7,
    learning_rate=0.05,
    num_leaves=63,
    class_weight='balanced',
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_samples=50,
    random_state=42,
    n_jobs=-1,
    verbose=-1
)
lgbm = evaluate_model('LightGBM', lgbm, X_train_sc, y_train, X_val_sc, y_val)

In [ ]:
# ─── 5.4 CatBoost ─────────────────────────────────────────────────────────────
# Handles imbalance well; robust out of the box
cat = CatBoostClassifier(
    iterations=500,
    depth=6,
    learning_rate=0.05,
    auto_class_weights='Balanced',
    eval_metric='F1',
    random_seed=42,
    verbose=0
)
cat = evaluate_model('CatBoost', cat, X_train_sc, y_train, X_val_sc, y_val)

## 6. Neural Network

In [ ]:
# ─── 6.1 Build Neural Network ─────────────────────────────────────────────────
# Tabular NN: BatchNorm + Dropout for regularisation
# Class weights passed to compensate for imbalance

def build_nn(input_dim):
    inp = keras.Input(shape=(input_dim,))
    x = layers.Dense(256, activation='relu')(inp)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(64, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.2)(x)
    x = layers.Dense(32, activation='relu')(x)
    out = layers.Dense(1, activation='sigmoid')(x)
    model = keras.Model(inp, out)
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss='binary_crossentropy',
        metrics=['accuracy', keras.metrics.AUC(name='auc'),
                 keras.metrics.Precision(name='precision'),
                 keras.metrics.Recall(name='recall')]
    )
    return model

nn_model = build_nn(X_train_sm.shape[1])
nn_model.summary()

In [ ]:
# ─── 6.2 Train Neural Network ─────────────────────────────────────────────────
class_weights_nn = {0: 1.0, 1: float(pos_weight)}

callbacks = [
    keras.callbacks.EarlyStopping(monitor='val_auc', patience=5,
                                   restore_best_weights=True, mode='max'),
    keras.callbacks.ReduceLROnPlateau(monitor='val_auc', factor=0.5,
                                       patience=3, mode='max', verbose=1)
]

history = nn_model.fit(
    X_train_sm, y_train_sm,
    validation_data=(X_val_sc, y_val),
    epochs=30,
    batch_size=2048,
    class_weight=class_weights_nn,
    callbacks=callbacks,
    verbose=1
)

In [ ]:
# ─── 6.3 Training History ─────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(history.history['loss'], label='Train Loss')
axes[0].plot(history.history['val_loss'], label='Val Loss')
axes[0].set_title('Training vs Validation Loss')
axes[0].set_xlabel('Epoch'); axes[0].legend()

axes[1].plot(history.history['auc'], label='Train AUC')
axes[1].plot(history.history['val_auc'], label='Val AUC')
axes[1].set_title('Training vs Validation AUC')
axes[1].set_xlabel('Epoch'); axes[1].legend()

plt.suptitle('Neural Network Training History', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('nn_training_history.png', dpi=150, bbox_inches='tight')
plt.show()

# Evaluate NN
nn_prob  = nn_model.predict(X_val_sc, verbose=0).ravel()

# Threshold tuning for F1
best_t, best_f1 = 0.5, 0
for t in np.arange(0.1, 0.9, 0.01):
    preds = (nn_prob >= t).astype(int)
    f = f1_score(y_val, preds, zero_division=0)
    if f > best_f1:
        best_f1, best_t = f, t

print(f'Best threshold: {best_t:.2f}  |  Best F1: {best_f1:.4f}')
nn_pred = (nn_prob >= best_t).astype(int)

acc  = accuracy_score(y_val, nn_pred)
f1   = f1_score(y_val, nn_pred, zero_division=0)
f1_macro = f1_score(y_val, nn_pred, average='macro', zero_division=0)

results_log.append({
    'Model': 'Neural Network',
    'Accuracy': acc,
    'Precision': precision_score(y_val, nn_pred, zero_division=0),
    'Recall': recall_score(y_val, nn_pred, zero_division=0),
    'F1 (anomaly)': f1,
    'F1 (macro)': f1_macro
})

print('\n=== Neural Network ===')
print(classification_report(y_val, nn_pred,
                            target_names=['Normal(0)', 'Anomaly(1)'],
                            zero_division=0))

## 7. Model Comparison

In [ ]:
# ─── Summary Table ────────────────────────────────────────────────────────────
results_df = pd.DataFrame(results_log).sort_values('F1 (macro)', ascending=False)
results_df = results_df.reset_index(drop=True)
results_df.index += 1

# Style the table
styled = results_df.style.background_gradient(
    subset=['F1 (macro)', 'F1 (anomaly)', 'Recall'], cmap='RdYlGn'
).format({
    'Accuracy': '{:.4f}', 'Precision': '{:.4f}',
    'Recall': '{:.4f}', 'F1 (anomaly)': '{:.4f}', 'F1 (macro)': '{:.4f}'
})

print('=== MODEL COMPARISON (sorted by Macro F1) ===')
display(styled)

In [ ]:
# ─── Bar chart comparison ─────────────────────────────────────────────────────
metrics = ['Accuracy', 'Precision', 'Recall', 'F1 (anomaly)', 'F1 (macro)']
x = np.arange(len(results_df))
width = 0.15
colors = ['#3498db', '#2ecc71', '#e74c3c', '#f39c12', '#9b59b6']

fig, ax = plt.subplots(figsize=(16, 6))
for i, (metric, color) in enumerate(zip(metrics, colors)):
    ax.bar(x + i*width, results_df[metric], width, label=metric, color=color, alpha=0.85)

ax.set_xticks(x + width*2)
ax.set_xticklabels(results_df['Model'], rotation=25, ha='right', fontsize=10)
ax.set_ylabel('Score')
ax.set_ylim(0, 1.1)
ax.set_title('Model Performance Comparison', fontsize=14, fontweight='bold')
ax.legend(loc='upper right')
ax.axhline(0.9, color='gray', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Cross-Validation (Robustness Check)

In [ ]:
# ─── 5-Fold Stratified CV on best model ───────────────────────────────────────
best_model_name = results_df.iloc[0]['Model']
print(f'Best model by Macro F1: {best_model_name}')

# Use LightGBM for CV (fast and usually top performer)
cv_model = LGBMClassifier(
    n_estimators=300, max_depth=7, learning_rate=0.05,
    num_leaves=63, class_weight='balanced',
    subsample=0.8, colsample_bytree=0.8,
    min_child_samples=50, random_state=42,
    n_jobs=-1, verbose=-1
)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_f1_scores = []

print('Running 5-Fold Stratified Cross-Validation...')
for fold, (tr_idx, vl_idx) in enumerate(skf.split(X, y), 1):
    Xf_tr, Xf_vl = X[tr_idx], X[vl_idx]
    yf_tr, yf_vl = y[tr_idx], y[vl_idx]

    # Scale
    sc_f = StandardScaler()
    Xf_tr = sc_f.fit_transform(Xf_tr)
    Xf_vl = sc_f.transform(Xf_vl)

    cv_model.fit(Xf_tr, yf_tr)
    preds = cv_model.predict(Xf_vl)
    f1_m = f1_score(yf_vl, preds, average='macro', zero_division=0)
    cv_f1_scores.append(f1_m)
    print(f'  Fold {fold}: Macro F1 = {f1_m:.4f}')

print(f'\nMean CV Macro F1 : {np.mean(cv_f1_scores):.4f}')
print(f'Std  CV Macro F1 : {np.std(cv_f1_scores):.4f}')
print('✅ Low std = robust, stable model')

## 9. Generate Final Submission

In [ ]:
# ─── Ensemble: Average probabilities from top models ──────────────────────────
# Combining LightGBM + XGBoost + CatBoost gives more robust predictions

prob_lgbm = lgbm.predict_proba(X_test_sc)[:, 1]
prob_xgb  = xgb.predict_proba(X_test_sc)[:, 1]
prob_cat  = cat.predict_proba(X_test_sc)[:, 1]
prob_nn   = nn_model.predict(X_test_sc, verbose=0).ravel()

# Weighted ensemble (give more weight to tree models)
ensemble_prob = (0.30 * prob_lgbm +
                 0.30 * prob_xgb  +
                 0.25 * prob_cat  +
                 0.15 * prob_nn)

# Threshold tuning on validation set
val_lgbm = lgbm.predict_proba(X_val_sc)[:, 1]
val_xgb  = xgb.predict_proba(X_val_sc)[:, 1]
val_cat  = cat.predict_proba(X_val_sc)[:, 1]
val_nn   = nn_model.predict(X_val_sc, verbose=0).ravel()

val_ensemble = (0.30 * val_lgbm + 0.30 * val_xgb +
                0.25 * val_cat  + 0.15 * val_nn)

best_t_ens, best_f1_ens = 0.5, 0
for t in np.arange(0.05, 0.95, 0.01):
    preds = (val_ensemble >= t).astype(int)
    f = f1_score(y_val, preds, average='macro', zero_division=0)
    if f > best_f1_ens:
        best_f1_ens, best_t_ens = f, t

print(f'Best ensemble threshold: {best_t_ens:.2f}')
print(f'Best ensemble Macro F1 on validation: {best_f1_ens:.4f}')

# Final predictions
final_preds = (ensemble_prob >= best_t_ens).astype(int)

print(f'\nFinal predictions — Normal: {(final_preds==0).sum():,}  |  Anomaly: {(final_preds==1).sum():,}')
print(f'Anomaly rate: {final_preds.mean()*100:.2f}%')

In [ ]:
# ─── Create submission file ───────────────────────────────────────────────────
submission = pd.DataFrame({
    'ID': test_ids,
    'target': final_preds.astype(str)   # must be string '0' or '1'
})

submission.to_csv('submission.csv', index=False)
print('✅  submission.csv saved!')
print(submission['target'].value_counts())
submission.head(10)

## 10. Key Insights & Conclusions

### Dataset Summary
- **1.64M training rows**, 5 sensor features (X1–X5) + Date
- **Severe class imbalance**: only **0.86% anomalies** → addressed via SMOTE + class weights
- **X3 & X4** contain exponential-scale values → log-transformed for classical models

### Feature Engineering
| Feature | Rationale |
|---|---|
| `X3_log`, `X4_log` | Compress exponential scale for linear models |
| `X4_extreme`, `X3_extreme` | Binary flag for extreme sensor readings |
| `X1*X2`, `X1*X5` | Sensor interaction signals |
| Temporal (`month`, `quarter`, `day_of_week`) | Anomalies may follow seasonal patterns |

### Model Strategy
- **Classical models** (LR, DT, KNN, SVM): baseline comparison with `class_weight='balanced'`
- **Ensemble models** (RF, XGBoost, LightGBM, CatBoost): boosted trees handle imbalance best
- **Neural Network**: deep tabular model with BatchNorm + Dropout
- **Final prediction**: weighted ensemble of LGBM + XGBoost + CatBoost + NN with optimised threshold

### Imbalance Handling
1. `SMOTE` — synthetic oversampling on training set (10% ratio)
2. `class_weight='balanced'` in tree models
3. `scale_pos_weight` in XGBoost
4. Threshold tuning on validation set for best macro F1